In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

In [3]:
sys.argv = ['-f'] + ["MIC", "-a", "cpu"]

In [4]:
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [5]:
from entry import * 

/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import importlib
import entry
import logging
importlib.reload(logging)
importlib.reload(entry)

<module 'entry' from '/beegfs/homes/pwiesenbach/BertGCN/entry.py'>

In [12]:
from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from operator import itemgetter
from collections import Counter
from clinic_datasets import CleanClinicDataset
import pickle
from torch.utils.data import Subset

In [13]:
dataset_file = Path("data") / f"medindcls_{args.bertmodel}_{args.doclevel}.json"
if not dataset_file.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, task="MIC", doclevel=args.doclevel, clean=False)
    with open(dataset_file, "wb") as f:
        print(f"Saving dataset under {dataset_file}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)
    
def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

Loading dataset from: data/medindcls_medbert_letter.json


In [17]:
MODELNAME = Path(PRETRAINEDMODEL).stem
if args.data == "MIC":
    DATASET = "med_indication_all_RF_diag"
    if args.testunklar:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}_testunklar"
    else:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}"
    BERTSAVEDIR = Path(f"models/finetuned/{args.doclevel}")
    if args.testunklar:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_testunklar_best.pt")
    else:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_best.pt")
elif args.data == "CSC":
    DATASET = "CARDIODE400_main"
    DATASETPATH =  Path("data") / f"ind.{DATASET}"
    BERTPATH = Path("models/finetuned/gbert-base_CARDIODE400_main_best.pt")

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

16104981.51778602

In [14]:
random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [18]:
first_order_adj = adj @ adj
first_order_adj = first_order_adj.toarray()

In [19]:
top_n_input = 10
top_input_test_rel_nodes = np.argpartition(first_order_adj[test_mask][:, doc_mask], -top_n_input)[:, -top_n_input:]
top_input_test_rel_nodes = np.vectorize(map_to_idx)(top_input_test_rel_nodes)
top_input_test_rel_nodes.max(), top_input_test_rel_nodes.shape

(2688, (540, 10))

In [20]:
input_df = pd.DataFrame(top_input_test_rel_nodes, index=test_idx)

In [21]:
input_df

,0,1,2,3,4,5,6,7,8,9
2298,2297,2296,2295,2300,2292,2293,2291,2298,2294,2299
2130,22,24,23,2130,2134,2136,2131,2135,2133,2132
2559,2563,2558,2561,2560,2559,2556,2557,2565,2562,2564
1046,1051,1049,1052,1046,1045,1048,1047,1050,1053,1044
1991,80,75,79,82,1990,1991,83,1992,76,84
...,...,...,...,...,...,...,...,...,...,...
2094,550,1603,551,1600,1601,2094,2095,2092,1602,2093
1060,1912,1061,1057,1055,1911,1059,1060,1058,1054,1056
165,1139,1133,1136,1140,161,166,162,164,165,163
1722,1731,1730,1733,1721,1726,1736,1729,1723,1722,1735


In [22]:
input_df = pd.melt(input_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [23]:
input_df

,id,rel_id
0,2298,2297
1,2130,22
2,2559,2563
3,1046,1051
4,1991,80
...,...,...
5395,2094,2093
5396,1060,1056
5397,165,163
5398,1722,1735


In [24]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in input_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in input_df.rel_id])
input_df["label"] = labels
input_df["rel_label"] = rel_labels

In [25]:
input_df

,id,rel_id,label,rel_label
0,2298,2297,Blutdrucksenker_beides,Cholesterinsenker_KHK
1,2130,22,Blutdrucksenker_Herzschw,Blutdrucksenker_Herzschw
2,2559,2563,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck
3,1046,1051,Cholesterinsenker_unklar,Blutdrucksenker_beides
4,1991,80,Blutdrucksenker_unklar,Cholesterinsenker_KHK
...,...,...,...,...
5395,2094,2093,Blutdrucksenker_Blutdruck,Blutdrucksenker_beides
5396,1060,1056,Blutdrucksenker_beides,Blutdrucksenker_beides
5397,165,163,Cholesterinsenker_beides,Blutdrucksenker_Blutdruck
5398,1722,1735,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck


In [26]:
input_source_df = input_df[["id", "label"]].drop_duplicates()
input_target_df = input_df[["rel_id", "rel_label"]].drop_duplicates()
input_node_df = pd.concat([input_source_df, input_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [27]:
input_G=nx.from_pandas_edgelist(input_df, "id", 'rel_id')

In [28]:
input_id_df = input_df[["id", "label"]]
input_rel_df = input_df[["rel_id", "rel_label"]]

In [29]:
new_columns = ["id", "label"]
input_id_df.columns = new_columns
input_rel_df.columns = new_columns

In [30]:
input_id_rel_df = pd.concat([input_id_df, input_rel_df], ignore_index=True).drop_duplicates()
input_id2label = dict(zip(input_id_rel_df.id, input_id_rel_df.label))
nx.set_node_attributes(input_G, input_id2label, "label")

In [31]:
input_id2text = {id: dataset.texts[id] for id in input_id_rel_df.id}
nx.set_node_attributes(input_G, input_id2text, "text")

In [32]:
input_id2drug = {node: input_G.nodes()[node]["text"].split(" ")[1] for node in input_G.nodes()}
nx.set_node_attributes(input_G, input_id2drug, "drug")

In [33]:
nx.is_connected(input_G), nx.number_connected_components(input_G)

(False, 62)

In [34]:
components = nx.connected_components(input_G)
largest_component = max(components, key=len)
subgraph = input_G.subgraph(largest_component)
diameter = nx.diameter(subgraph)
print("Network diameter of largest component:", diameter)

Network diameter of largest component: 23


In [38]:
components = [x for x in nx.connected_components(input_G)]
[len(x) for x in components]

[31,
 1208,
 10,
 14,
 10,
 74,
 18,
 12,
 10,
 10,
 10,
 11,
 38,
 10,
 19,
 39,
 10,
 30,
 18,
 19,
 10,
 11,
 26,
 11,
 17,
 10,
 10,
 10,
 10,
 10,
 23,
 10,
 12,
 11,
 10,
 12,
 12,
 10,
 12,
 10,
 10,
 10,
 10,
 10,
 10,
 16,
 10,
 10,
 11,
 11,
 10,
 11,
 13,
 10,
 11,
 10,
 10,
 10,
 11,
 10,
 10,
 10]

In [40]:
[Counter([input_G.nodes()[x]["drug"] for x in com]).most_common()[0] for com in components]

[('Simvastatin', 6),
 ('Ramipril', 100),
 ('Sortis', 1),
 ('Pravasin', 3),
 ('Torem', 1),
 ('Metoprololsuccinat', 6),
 ('Simvastatin', 4),
 ('Doxazosin', 1),
 ('Valsartan', 2),
 ('Bisoprolol', 2),
 ('Ramipril', 2),
 ('Simvastatin', 2),
 ('Delix', 3),
 ('Concor', 1),
 ('Ramipril', 3),
 ('Pravastatin', 6),
 ('Metodura', 1),
 ('Sortis', 5),
 ('Sortis', 2),
 ('Torasemid', 2),
 ('Lantus', 1),
 ('Amlodipin', 2),
 ('Ramipril', 3),
 ('Protaphane', 1),
 ('Beloc', 2),
 ('Simvastatin', 2),
 ('Aldactone', 1),
 ('Isoptin', 2),
 ('Carvedilol', 1),
 ('HCT', 2),
 ('Delix', 3),
 ('Sevikar', 2),
 ('Inegy', 2),
 ('Esidrix', 1),
 ('Sortis', 1),
 ('HCT', 3),
 ('Toujeo', 2),
 ('Sortis', 1),
 ('Siofor', 2),
 ('Dilatrend', 1),
 ('HCT', 2),
 ('Ramipril', 2),
 ('Beloc', 1),
 ('Torasemid', 2),
 ('Beloc', 2),
 ('Beloc', 3),
 ('Candesartan', 2),
 ('Beloc', 1),
 ('Pravasin', 1),
 ('Carmen', 1),
 ('Atenolol', 1),
 ('Bisoprolol', 2),
 ('Belok', 2),
 ('Amlobeta', 2),
 ('Hydrochlorothiazid', 1),
 ('Metroprolol', 2),
 (

In [41]:
[Counter([input_G.nodes()[x]["label"] for x in com]).most_common()[0] for com in components]

[('Blutdrucksenker_Blutdruck', 14),
 ('Blutdrucksenker_Blutdruck', 383),
 ('Blutdrucksenker_Blutdruck', 6),
 ('Blutdrucksenker_Blutdruck', 8),
 ('Blutdrucksenker_unklar', 5),
 ('Blutdrucksenker_beides', 38),
 ('Blutdrucksenker_beides', 8),
 ('Blutdrucksenker_Blutdruck', 6),
 ('Blutdrucksenker_beides', 5),
 ('Blutdrucksenker_Blutdruck', 6),
 ('Blutdrucksenker_Blutdruck', 6),
 ('Blutdrucksenker_beides', 6),
 ('Blutdrucksenker_beides', 14),
 ('Blutdrucksenker_beides', 8),
 ('Blutdrucksenker_beides', 11),
 ('Blutdrucksenker_unklar', 8),
 ('Blutdrucksenker_beides', 4),
 ('Blutdrucksenker_beides', 15),
 ('Blutdrucksenker_beides', 8),
 ('Blutdrucksenker_beides', 4),
 ('Blutdrucksenker_Blutdruck', 6),
 ('Blutdrucksenker_Blutdruck', 9),
 ('Blutdrucksenker_Blutdruck', 8),
 ('Blutdrucksenker_beides', 6),
 ('Blutdrucksenker_Blutdruck', 12),
 ('Blutdrucksenker_Blutdruck', 8),
 ('Blutdrucksenker_Herzschw', 6),
 ('Blutdrucksenker_Blutdruck', 10),
 ('Blutdrucksenker_Blutdruck', 9),
 ('Blutdrucksenker_

In [35]:
input_triadic_closure = nx.transitivity(input_G)
input_triadic_closure

0.3412579302817382

In [36]:
input_degree_dict = dict(input_G.degree(input_G.nodes()))
nx.set_node_attributes(input_G, input_degree_dict, 'degree')
input_sorted_degree = sorted(input_degree_dict.items(), key=itemgetter(1), reverse=True)

input_sorted_degree[:10]

[(844, 31),
 (842, 29),
 (1975, 23),
 (323, 23),
 (1548, 23),
 (843, 22),
 (321, 20),
 (1991, 19),
 (322, 19),
 (1550, 19)]

In [28]:
input_betweenness_dict = nx.betweenness_centrality(input_G)
nx.set_node_attributes(input_G, input_betweenness_dict, 'betweenness')
input_sorted_betweenness = sorted(input_betweenness_dict.items(), key=itemgetter(1), reverse=True)

input_sorted_betweenness[:10]

[(842, 0.13513169394261643),
 (1352, 0.08667820433708456),
 (2525, 0.08192994592530023),
 (1548, 0.07882937665922923),
 (1912, 0.07776295288016025),
 (844, 0.059915971770387905),
 (1516, 0.04355991294257442),
 (72, 0.04272085167279535),
 (1303, 0.040922763603986556),
 (1911, 0.040120027737913184)]

In [29]:
input_communities = nx.community.greedy_modularity_communities(input_G)
input_modularity_dict = {}
for i, c in enumerate(input_communities):
    for name in c:
        input_modularity_dict[name] = i
nx.set_node_attributes(input_G, input_modularity_dict, 'community')

len(input_communities)

82

In [30]:
[Counter([input_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in input_communities[:10]]

[0.27702702702702703,
 0.3130434782608696,
 0.40540540540540543,
 0.34545454545454546,
 0.40186915887850466,
 0.38461538461538464,
 0.5135135135135135,
 0.3698630136986301,
 0.42857142857142855,
 0.3684210526315789]

In [31]:
[Counter([input_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in input_communities[:10]]

[('Blutdrucksenker_beides', 41),
 ('Blutdrucksenker_beides', 36),
 ('Blutdrucksenker_Blutdruck', 45),
 ('Blutdrucksenker_beides', 38),
 ('Blutdrucksenker_Blutdruck', 43),
 ('Blutdrucksenker_beides', 40),
 ('Blutdrucksenker_beides', 38),
 ('Blutdrucksenker_Blutdruck', 27),
 ('Blutdrucksenker_beides', 27),
 ('Blutdrucksenker_beides', 21)]

In [32]:
train_count = 0
val_count = 0
test_count = 0

for node in input_G.nodes():
    if node not in test_idx: 
        continue
    for n in input_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6710778338847292, 0.0, 0.32892216611527075)